In [ ]:
!pip install -U datasets transformers evaluate accelerate tdqm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.2 MB/s eta 0:00:0

Fine-tuninig BERT model on the SQuAD dataset, which consists of questions posed by crowdworkers on a set of Wikipedia articles.

In [ ]:
from datasets import load_dataset

ds = load_dataset("squad")
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

### EDA

**Train dataset**

In [ ]:
ds["train"].column_names

['id', 'title', 'context', 'question', 'answers']

In [ ]:
ds["train"].features

{'id': Value(dtype='string', id=None),
 'title': Value(dtype='string', id=None),
 'context': Value(dtype='string', id=None),
 'question': Value(dtype='string', id=None),
 'answers': Sequence(feature={'text': Value(dtype='string', id=None), 'answer_start': Value(dtype='int32', id=None)}, length=-1, id=None)}

In [ ]:
ds["train"][0]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

In [ ]:
ds["train"][0]["context"]

'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [ ]:
len(ds["train"][0]["context"].split())

124

In [ ]:
len(ds["train"][0]["context"])

695

In [ ]:
ds["train"][0]["question"]

'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'

In [ ]:
ds["train"][0]["answers"] # dict: `text` is a list of answers, `answer_start` is its starting character index in context

{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}

In [ ]:
ds["train"][0]["answers"]["text"][0]

'Saint Bernadette Soubirous'

In [ ]:
ds["train"][0]["answers"]["answer_start"][0]

515

In [ ]:
len(ds["train"][0]["answers"]["text"][0])

26

In [ ]:
ds["train"][0]["context"][515:515+26]  # extract the answer from context

'Saint Bernadette Soubirous'

In [ ]:
ds["train"][0]["title"]  # not required for this task

'University_of_Notre_Dame'

In [ ]:
ds["train"][0]["id"] # unique hash id for each example in the data

'5733be284776f41900661182'

**Val dataset**

In [ ]:
ds["validation"][0]

{'id': '56be4db0acb8001400a502ec',
 'title': 'Super_Bowl_50',
 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.',
 'question': 'Which NFL team represented the AFC at Super Bowl 50?',
 'answers': {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],


In [ ]:
ds["validation"][0]["context"]

'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.'

In [ ]:
len(ds["validation"][0]["context"])

775

In [ ]:
ds["validation"][0]["question"]

'Which NFL team represented the AFC at Super Bowl 50?'

In [ ]:
ds["validation"][0]["answers"]  # observe that validation set has more than one answer for a single question

{'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],
 'answer_start': [177, 177, 177]}

In [ ]:
ds["validation"][1]["answers"]

{'text': ['Carolina Panthers', 'Carolina Panthers', 'Carolina Panthers'],
 'answer_start': [249, 249, 249]}

In [ ]:
ds["validation"][2]

{'id': '56be4db0acb8001400a502ee',
 'title': 'Super_Bowl_50',
 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.',
 'question': 'Where did Super Bowl 50 take place?',
 'answers': {'text': ['Santa Clara, California',
   "Levi's Stadium",
   "Levi's Stadium in th

In [ ]:
ds["validation"][2]["answers"] # observe that validation set has more than one answer for a single question

{'text': ['Santa Clara, California',
  "Levi's Stadium",
  "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California."],
 'answer_start': [403, 355, 355]}

### How BERT trains for Q&A dataset like Squad

When using **BERT for Question Answering (QA)** (like in SQuAD), both the **question and the context** are processed **together** as a single input sequence, and **attention operates across both** — not separately.

---

#### 1. **Input Format to BERT in QA**

We concatenate the **question** and **context** like this: (Tokenizer does this for us. Just need to send both question and context separated by comma to tokenizer)

```text
[CLS] question [SEP] context [SEP]
```

Then BERT receives:

* A single sequence of **tokens** (tokenized with WordPiece/BPE).
* **Segment embeddings**: to tell BERT which part is the question (`segment A`) and which is the context (`segment B`).(=>`token_type_ids`)
* **Position embeddings**: for token positions.

---

#### 2. **Self-Attention in BERT**

#### What happens:

* Every token in the input (question **and** context) can **attend to every other token** in the sequence — this is **bidirectional self-attention**.

So:

* **Question tokens attend to context**
* **Context tokens attend to question**
* **Tokens within each part attend to each other**

#### Why this is powerful:

* BERT can learn **which part of the context answers which part of the question**, using deep attention layers.
* Example: If your question is "Who wrote Hamlet?" and the context says "Shakespeare was a famous playwright who wrote Hamlet", BERT learns through attention that *"who wrote"* in the question aligns with *"Shakespeare"* in the context.

---

#### 3. **Prediction Head**

BERT is usually fine-tuned for QA with a simple head:

* For each token, we predict:

  * **Start position** of the answer
  * **End position** of the answer

That is:

```python
start_logits, end_logits = BERT_output → linear layer → softmax over sequence length
```

The most likely span between start and end is taken as the answer.

---


| Step          | Action                                                          |
| ------------- | --------------------------------------------------------------- |
| **Input**     | `[CLS] question [SEP] context [SEP]`                            |
| **Embedding** | Token + Position + Segment embeddings                           |
| **Attention** | Full self-attention over **both question and context**          |
| **Output**    | For each token → predict start and end positions in the context |

---
**Output**

> We predict **start and end logits for each token** in the input sequence (which includes both question and context).
> But at inference time, **we only consider tokens in the context** when selecting the final answer span.

---
#### For eg:

You have this input:

```
Who wrote Hamlet? Shakespeare was a famous playwright who wrote Hamlet.
```

This gets tokenized into something like:

```
[CLS] Who wrote Hamlet ? [SEP] Shakespeare was a famous playwright who wrote Hamlet . [SEP]
```

Let's say that results in **50 tokens** total.

---

#### What BERT Does:

* BERT outputs a hidden vector for **each of those 50 tokens**.
* Then, a small feed-forward layer is applied to each token's output to get:

  * **One start logit**
  * **One end logit**

So you end up with:

* `start_logits`: shape `(seq_len,)` → scores for each token being the **start** of the answer
* `end_logits`: shape `(seq_len,)` → scores for each token being the **end** of the answer

---

#### But what about the question tokens?

**Why score the question tokens?**

* During training, we include **all tokens** because it simplifies the computation.
* During **inference**, we **ignore logits for tokens that belong to the question** or `[CLS]` and `[SEP]` — we only select start and end positions from the **context** part.

---

#### Final Answer Span:

```python
best_start = argmax(start_logits[context_only])
best_end = argmax(end_logits[context_only])
```

* The model may optionally try all `(start, end)` combinations within a window (e.g., max 15 tokens apart) to find the best span.

---


| Aspect               | Behavior                            |
| -------------------- | ----------------------------------- |
| Prediction           | One start + end score **per token** |
| Applies to           | **All tokens** (question + context) |
| Answer selected from | **Context tokens only**             |


Basically BERT model for Q&A outputs start logits and end logits for each token which are nothing but probs for each token that the how likely that token is the start of the answer or end of the answer. See it as binary classfication for each token in the sequence.

---


## What do start and end positions mean?

They are indices of tokens in the input sequence.

For example, given this tokenized input:

```
[CLS] Who wrote Hamlet? [SEP] Shakespeare was a famous playwright who wrote Hamlet. [SEP]
```

If the answer is "Shakespeare", and it starts at token index **6**, ends at **6**, your target (true) labels are:

```python
start_position = 6
end_position = 6
```

---

## 2. **Model Output:**

From BERT, after the final layer and a classifier head:

```python
start_logits: Tensor of shape (sequence_length,)
end_logits: Tensor of shape (sequence_length,)
```

Each token gets:

* One logit for being the **start**
* One logit for being the **end**

These are **raw scores**, not probabilities yet.

---

## 3. **Convert logits to probabilities (during evaluation, not training):**

```python
start_probs = softmax(start_logits)
end_probs = softmax(end_logits)
```

But for training, you usually apply **cross-entropy loss directly to the logits** (not the softmax output as the loss apply softmax behind the scenes).

---

## 4. **How is the loss calculated?**

You compute **two separate cross-entropy losses**:

```python
start_loss = cross_entropy(start_logits, start_position)
end_loss = cross_entropy(end_logits, end_position)
total_loss = (start_loss + end_loss) / 2
```

Each loss compares the predicted logits against the **true index** of the start or end token.

### So what's happening?

* **start\_logits** is a vector of length `seq_len`
* **start\_position** is a single integer index (the correct token)

The cross-entropy loss function:

* Applies `log(softmax(logits))` internally
* Then compares it with the correct class index


The **negative log-likelihood** is computed **at the true index** (the true start or end token), not subtracted *from* it.

> The **cross-entropy loss** selects the log-probability of the correct token and returns its **negative log** (hence: Negative Log Likelihood = NLL).

Suppose your model produces this output for **start logits** (after softmax):

| Token Index | Token           | Probability (softmax output) |
| ----------- | --------------- | ---------------------------- |
| 0           | \[CLS]          | 0.01                         |
| 1           | Who             | 0.02                         |
| 2           | wrote           | 0.03                         |
| 3           | Hamlet          | 0.01                         |
| 4           | ?               | 0.01                         |
| 5           | \[SEP]          | 0.01                         |
| 6           | **Shakespeare** | **0.85** ← correct index     |
| ...         | ...             | ...                          |

### True start index: `6` (for “Shakespeare”)

---

## Cross Entropy Loss:

In PyTorch (or TensorFlow), this is done like:

```python
loss = -log(probability_at_true_index)
```

So:

```python
loss = -log(0.85) ≈ 0.1625
```

No subtraction from the index. The index is used to **look up the predicted probability**, and then the log of that value is used.

---

## Internally (in code):

When using PyTorch:

```python
import torch.nn.functional as F

loss = F.cross_entropy(start_logits, start_positions)
```

* `start_logits`: shape `[batch_size, seq_len]`
* `start_positions`: shape `[batch_size]` with integer indices
* `cross_entropy`:

  1. Applies softmax to `start_logits` to get probabilities
  2. Picks the log-probability at the `start_positions`
  3. Takes the **negative** of that log-probability

This is exactly the same as Negative Log-Likelihood Loss (NLLLoss), applied over the softmax.

---

## Summary:

| Step              | Description                                           |
| ----------------- | ----------------------------------------------------- |
| **Target Labels** | Indices of start and end tokens (integers)            |
| **Model Outputs** | Logits for each token (before softmax)                |
| **Loss Function** | Cross-entropy between predicted logits and true index |
| **Final Loss**    | Average of start and end position loss                |


### Preprocessing

#### Tokenizer

In [ ]:
# We use bert-base-cased tokenizer to tokenizer both context and question, as we need to send these both to our model.
from transformers import AutoTokenizer

checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer

BertTokenizerFast(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
tokenizer.is_fast

True

In [ ]:
# Test
tokenizer("I play football sometimes")

{'input_ids': [101, 146, 1505, 1709, 2121, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer.decode(tokenizer("I play football sometimes")["input_ids"])

'[CLS] I play football sometimes [SEP]'

In [ ]:
# Test
question = ds["train"][0:3]["question"]
context = ds["train"][0:3]["context"]

inputs = tokenizer(question, context)
inputs

{'input_ids': [[101, 1706, 2292, 1225, 1103, 6567, 2090, 9273, 2845, 1107, 8109, 1107, 10111, 20500, 1699, 136, 102, 22182, 1193, 117, 1103, 1278, 1144, 170, 2336, 1959, 119, 1335, 4184, 1103, 4304, 4334, 112, 188, 2284, 10945, 1110, 170, 5404, 5921, 1104, 1103, 6567, 2090, 119, 13301, 1107, 1524, 1104, 1103, 4304, 4334, 1105, 4749, 1122, 117, 1110, 170, 7335, 5921, 1104, 4028, 1114, 1739, 1146, 14089, 5591, 1114, 1103, 7051, 107, 159, 21462, 1566, 24930, 2508, 152, 1306, 3965, 107, 119, 5893, 1106, 1103, 4304, 4334, 1110, 1103, 19349, 1104, 1103, 11373, 4641, 119, 13301, 1481, 1103, 171, 17506, 9538, 1110, 1103, 144, 10595, 2430, 117, 170, 14789, 1282, 1104, 8070, 1105, 9284, 119, 1135, 1110, 170, 16498, 1104, 1103, 176, 10595, 2430, 1120, 10111, 20500, 117, 1699, 1187, 1103, 6567, 2090, 25153, 1193, 1691, 1106, 2216, 17666, 6397, 3786, 1573, 25422, 13149, 1107, 8109, 119, 1335, 1103, 1322, 1104, 1103, 1514, 2797, 113, 1105, 1107, 170, 2904, 1413, 1115, 8200, 1194, 124, 11739, 1105, 1

In [ ]:
inputs.keys()

KeysView({'input_ids': [[101, 1706, 2292, 1225, 1103, 6567, 2090, 9273, 2845, 1107, 8109, 1107, 10111, 20500, 1699, 136, 102, 22182, 1193, 117, 1103, 1278, 1144, 170, 2336, 1959, 119, 1335, 4184, 1103, 4304, 4334, 112, 188, 2284, 10945, 1110, 170, 5404, 5921, 1104, 1103, 6567, 2090, 119, 13301, 1107, 1524, 1104, 1103, 4304, 4334, 1105, 4749, 1122, 117, 1110, 170, 7335, 5921, 1104, 4028, 1114, 1739, 1146, 14089, 5591, 1114, 1103, 7051, 107, 159, 21462, 1566, 24930, 2508, 152, 1306, 3965, 107, 119, 5893, 1106, 1103, 4304, 4334, 1110, 1103, 19349, 1104, 1103, 11373, 4641, 119, 13301, 1481, 1103, 171, 17506, 9538, 1110, 1103, 144, 10595, 2430, 117, 170, 14789, 1282, 1104, 8070, 1105, 9284, 119, 1135, 1110, 170, 16498, 1104, 1103, 176, 10595, 2430, 1120, 10111, 20500, 117, 1699, 1187, 1103, 6567, 2090, 25153, 1193, 1691, 1106, 2216, 17666, 6397, 3786, 1573, 25422, 13149, 1107, 8109, 119, 1335, 1103, 1322, 1104, 1103, 1514, 2797, 113, 1105, 1107, 170, 2904, 1413, 1115, 8200, 1194, 124, 11739

In [ ]:
len(inputs["input_ids"])

3

In [ ]:
type(inputs["input_ids"])

list

In [ ]:
for input in inputs["input_ids"]:
    print(tokenizer.decode(input))  # [CLS] question [SEP] context [SEP]

[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building ' s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive ( and in a direct line that connects through 3 statues and the Gold Dome ), is a simple, modern stone statue of Mary. [SEP]
[CLS] What is in front of the Notre Dame Main Building? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building ' s gold dome is a golden statue of the Virgin Mary. Immediately in f

In [ ]:
for input in inputs["input_ids"]:
    print(len(input))
    print(tokenizer.decode(input))

181
[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building ' s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive ( and in a direct line that connects through 3 statues and the Gold Dome ), is a simple, modern stone statue of Mary. [SEP]
177
[CLS] What is in front of the Notre Dame Main Building? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building ' s gold dome is a golden statue of the Virgin Mary. Immediat

In [ ]:
for segment_ids in inputs["token_type_ids"]:
    print(segment_ids)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

#### Overflow sample mapping and offset mapping to handle long contexts

In [ ]:
inputs = tokenizer(question, context, max_length=100, truncation="only_second", stride=50, return_overflowing_tokens=True, return_offsets_mapping=True)

- `max_length` is max sequence length to specify how many tokens a model can process at once. If actual_seq_len < max_length, then it returns actual_seq_len tokens
- `return_overflowing_tokens` to also return all remaining chunks after specified max_length
- `stride` to get some overlapping tokens for each chunk to maintain connectivity between chunks
- `return_offsets_mapping` gives the token positions/offsets in questions and contexts, which are later used in further preprocessing steps
- `truncation="only_second"` helps us to over perform truncation and all the above steps on second sentence in the input which is nothing but our context. We dont do truncation or overflow on question.

In [ ]:
inputs

{'input_ids': [[101, 1706, 2292, 1225, 1103, 6567, 2090, 9273, 2845, 1107, 8109, 1107, 10111, 20500, 1699, 136, 102, 22182, 1193, 117, 1103, 1278, 1144, 170, 2336, 1959, 119, 1335, 4184, 1103, 4304, 4334, 112, 188, 2284, 10945, 1110, 170, 5404, 5921, 1104, 1103, 6567, 2090, 119, 13301, 1107, 1524, 1104, 1103, 4304, 4334, 1105, 4749, 1122, 117, 1110, 170, 7335, 5921, 1104, 4028, 1114, 1739, 1146, 14089, 5591, 1114, 1103, 7051, 107, 159, 21462, 1566, 24930, 2508, 152, 1306, 3965, 107, 119, 5893, 1106, 1103, 4304, 4334, 1110, 1103, 19349, 1104, 1103, 11373, 4641, 119, 13301, 1481, 1103, 171, 17506, 102], [101, 1706, 2292, 1225, 1103, 6567, 2090, 9273, 2845, 1107, 8109, 1107, 10111, 20500, 1699, 136, 102, 1103, 4304, 4334, 1105, 4749, 1122, 117, 1110, 170, 7335, 5921, 1104, 4028, 1114, 1739, 1146, 14089, 5591, 1114, 1103, 7051, 107, 159, 21462, 1566, 24930, 2508, 152, 1306, 3965, 107, 119, 5893, 1106, 1103, 4304, 4334, 1110, 1103, 19349, 1104, 1103, 11373, 4641, 119, 13301, 1481, 1103, 171

Observe that now each sample in input_ids (i.e a list) is divided in chunks (sublists). Each chunk is of length `max_len`

In [ ]:
len(inputs["input_ids"]) # length increased from 3 previously to 12 because of chunking

12

In [ ]:
for input in inputs["input_ids"]:
    print(len(input))
    print(tokenizer.decode(input))

# observe that we have same question for every chunk of its context because we only chunked and truncated second sequence
# also see how stride is making the context repeated in every chunk


100
[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building ' s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basi [SEP]
100
[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin [SEP]
100
[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Next to the Main Bui

In [ ]:
inputs.keys() # we got `offset_mapping` and `overflow_to_sample_mapping`

KeysView({'input_ids': [[101, 1706, 2292, 1225, 1103, 6567, 2090, 9273, 2845, 1107, 8109, 1107, 10111, 20500, 1699, 136, 102, 22182, 1193, 117, 1103, 1278, 1144, 170, 2336, 1959, 119, 1335, 4184, 1103, 4304, 4334, 112, 188, 2284, 10945, 1110, 170, 5404, 5921, 1104, 1103, 6567, 2090, 119, 13301, 1107, 1524, 1104, 1103, 4304, 4334, 1105, 4749, 1122, 117, 1110, 170, 7335, 5921, 1104, 4028, 1114, 1739, 1146, 14089, 5591, 1114, 1103, 7051, 107, 159, 21462, 1566, 24930, 2508, 152, 1306, 3965, 107, 119, 5893, 1106, 1103, 4304, 4334, 1110, 1103, 19349, 1104, 1103, 11373, 4641, 119, 13301, 1481, 1103, 171, 17506, 102], [101, 1706, 2292, 1225, 1103, 6567, 2090, 9273, 2845, 1107, 8109, 1107, 10111, 20500, 1699, 136, 102, 1103, 4304, 4334, 1105, 4749, 1122, 117, 1110, 170, 7335, 5921, 1104, 4028, 1114, 1739, 1146, 14089, 5591, 1114, 1103, 7051, 107, 159, 21462, 1566, 24930, 2508, 152, 1306, 3965, 107, 119, 5893, 1106, 1103, 4304, 4334, 1110, 1103, 19349, 1104, 1103, 11373, 4641, 119, 13301, 1481, 

- `overflow_to_sample_mapping` gives the mapping of each chunk to its original sample. Eg: [0, 0, 0, 1, 1] tells us that first 3 chunks belong original smaple(i.e row) with idx 0 and next 2 chunks belong to original sample(i.e row) with idx 1. Also note that each question`s context's chunk has its question repeated because we perform truncation and overflow only second sequence.
- `offset_mapping` gives the start char and end char position of each token of original sample, where as special tokens like [CLS] and [SEP] have offsets as (0, 0). offset_mapping[idx] gives you (start_char, end_char) for the token at idx. It provides a tuple (start_char_index, end_char_index) indicating which characters in the original input string that particular token corresponds to

In [ ]:
print(len(inputs["overflow_to_sample_mapping"]))

12


In [ ]:
print(inputs["overflow_to_sample_mapping"]) # tells each chunk belongs to which orignal sample

[0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]


In [ ]:
print(len(inputs["token_type_ids"]))

12


In [ ]:
print(inputs["token_type_ids"])

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
for segment_ids in inputs["token_type_ids"]:
    print(segment_ids)  # 0 tells that is it question i.e first seq (cls+seq+sep) and 1 tells that it is context i.e second seq (seq+sep)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [ ]:
len(inputs.tokens())  # only gives first chunk tokens, but can access others with index

100

In [ ]:
inputs.tokens()

['[CLS]',
 'To',
 'whom',
 'did',
 'the',
 'Virgin',
 'Mary',
 'allegedly',
 'appear',
 'in',
 '1858',
 'in',
 'Lou',
 '##rdes',
 'France',
 '?',
 '[SEP]',
 'Architectural',
 '##ly',
 ',',
 'the',
 'school',
 'has',
 'a',
 'Catholic',
 'character',
 '.',
 'At',
 '##op',
 'the',
 'Main',
 'Building',
 "'",
 's',
 'gold',
 'dome',
 'is',
 'a',
 'golden',
 'statue',
 'of',
 'the',
 'Virgin',
 'Mary',
 '.',
 'Immediately',
 'in',
 'front',
 'of',
 'the',
 'Main',
 'Building',
 'and',
 'facing',
 'it',
 ',',
 'is',
 'a',
 'copper',
 'statue',
 'of',
 'Christ',
 'with',
 'arms',
 'up',
 '##rai',
 '##sed',
 'with',
 'the',
 'legend',
 '"',
 'V',
 '##eni',
 '##te',
 'Ad',
 'Me',
 'O',
 '##m',
 '##nes',
 '"',
 '.',
 'Next',
 'to',
 'the',
 'Main',
 'Building',
 'is',
 'the',
 'Basilica',
 'of',
 'the',
 'Sacred',
 'Heart',
 '.',
 'Immediately',
 'behind',
 'the',
 'b',
 '##asi',
 '[SEP]']

In [ ]:
inputs.tokens(1) # second chunk tokens

['[CLS]',
 'To',
 'whom',
 'did',
 'the',
 'Virgin',
 'Mary',
 'allegedly',
 'appear',
 'in',
 '1858',
 'in',
 'Lou',
 '##rdes',
 'France',
 '?',
 '[SEP]',
 'the',
 'Main',
 'Building',
 'and',
 'facing',
 'it',
 ',',
 'is',
 'a',
 'copper',
 'statue',
 'of',
 'Christ',
 'with',
 'arms',
 'up',
 '##rai',
 '##sed',
 'with',
 'the',
 'legend',
 '"',
 'V',
 '##eni',
 '##te',
 'Ad',
 'Me',
 'O',
 '##m',
 '##nes',
 '"',
 '.',
 'Next',
 'to',
 'the',
 'Main',
 'Building',
 'is',
 'the',
 'Basilica',
 'of',
 'the',
 'Sacred',
 'Heart',
 '.',
 'Immediately',
 'behind',
 'the',
 'b',
 '##asi',
 '##lica',
 'is',
 'the',
 'G',
 '##rot',
 '##to',
 ',',
 'a',
 'Marian',
 'place',
 'of',
 'prayer',
 'and',
 'reflection',
 '.',
 'It',
 'is',
 'a',
 'replica',
 'of',
 'the',
 'g',
 '##rot',
 '##to',
 'at',
 'Lou',
 '##rdes',
 ',',
 'France',
 'where',
 'the',
 'Virgin',
 '[SEP]']

In [ ]:
inputs.sequence_ids() # token_type_ids with special tokens as None

[None,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 None,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 None]

In [ ]:
len(inputs.sequence_ids())

100

In [ ]:
inputs.sequence_ids(0)

[None,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 None,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 None]

In [ ]:
print(len(inputs["offset_mapping"]))

12


In [ ]:
print(inputs["offset_mapping"]) # offset mapping of each token in question and also context

[[(0, 0), (0, 2), (3, 7), (8, 11), (12, 15), (16, 22), (23, 27), (28, 37), (38, 44), (45, 47), (48, 52), (53, 55), (56, 59), (59, 63), (64, 70), (70, 71), (0, 0), (0, 13), (13, 15), (15, 16), (17, 20), (21, 27), (28, 31), (32, 33), (34, 42), (43, 52), (52, 53), (54, 56), (56, 58), (59, 62), (63, 67), (68, 76), (76, 77), (77, 78), (79, 83), (84, 88), (89, 91), (92, 93), (94, 100), (101, 107), (108, 110), (111, 114), (115, 121), (122, 126), (126, 127), (128, 139), (140, 142), (143, 148), (149, 151), (152, 155), (156, 160), (161, 169), (170, 173), (174, 180), (181, 183), (183, 184), (185, 187), (188, 189), (190, 196), (197, 203), (204, 206), (207, 213), (214, 218), (219, 223), (224, 226), (226, 229), (229, 232), (233, 237), (238, 241), (242, 248), (249, 250), (250, 251), (251, 254), (254, 256), (257, 259), (260, 262), (263, 264), (264, 265), (265, 268), (268, 269), (269, 270), (271, 275), (276, 278), (279, 282), (283, 287), (288, 296), (297, 299), (300, 303), (304, 312), (313, 315), (316,

In [ ]:
for offset_chunk in inputs["offset_mapping"]:
    print(offset_chunk) # (start char pos, end char pos) for each token

[(0, 0), (0, 2), (3, 7), (8, 11), (12, 15), (16, 22), (23, 27), (28, 37), (38, 44), (45, 47), (48, 52), (53, 55), (56, 59), (59, 63), (64, 70), (70, 71), (0, 0), (0, 13), (13, 15), (15, 16), (17, 20), (21, 27), (28, 31), (32, 33), (34, 42), (43, 52), (52, 53), (54, 56), (56, 58), (59, 62), (63, 67), (68, 76), (76, 77), (77, 78), (79, 83), (84, 88), (89, 91), (92, 93), (94, 100), (101, 107), (108, 110), (111, 114), (115, 121), (122, 126), (126, 127), (128, 139), (140, 142), (143, 148), (149, 151), (152, 155), (156, 160), (161, 169), (170, 173), (174, 180), (181, 183), (183, 184), (185, 187), (188, 189), (190, 196), (197, 203), (204, 206), (207, 213), (214, 218), (219, 223), (224, 226), (226, 229), (229, 232), (233, 237), (238, 241), (242, 248), (249, 250), (250, 251), (251, 254), (254, 256), (257, 259), (260, 262), (263, 264), (264, 265), (265, 268), (268, 269), (269, 270), (271, 275), (276, 278), (279, 282), (283, 287), (288, 296), (297, 299), (300, 303), (304, 312), (313, 315), (316, 

In [ ]:
print(len(inputs["attention_mask"]))

12


In [ ]:
print(inputs["attention_mask"])

[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
for chunk in inputs["attention_mask"]:
    print(input)

[101, 1109, 19349, 1104, 1103, 11373, 1762, 1120, 10360, 8022, 1110, 3148, 1106, 1134, 2401, 136, 102, 119, 1135, 1110, 170, 16498, 1104, 1103, 176, 10595, 2430, 1120, 10111, 20500, 117, 1699, 1187, 1103, 6567, 2090, 25153, 1193, 1691, 1106, 2216, 17666, 6397, 3786, 1573, 25422, 13149, 1107, 8109, 119, 1335, 1103, 1322, 1104, 1103, 1514, 2797, 113, 1105, 1107, 170, 2904, 1413, 1115, 8200, 1194, 124, 11739, 1105, 1103, 3487, 17917, 114, 117, 1110, 170, 3014, 117, 2030, 2576, 5921, 1104, 2090, 119, 102]
[101, 1109, 19349, 1104, 1103, 11373, 1762, 1120, 10360, 8022, 1110, 3148, 1106, 1134, 2401, 136, 102, 119, 1135, 1110, 170, 16498, 1104, 1103, 176, 10595, 2430, 1120, 10111, 20500, 117, 1699, 1187, 1103, 6567, 2090, 25153, 1193, 1691, 1106, 2216, 17666, 6397, 3786, 1573, 25422, 13149, 1107, 8109, 119, 1335, 1103, 1322, 1104, 1103, 1514, 2797, 113, 1105, 1107, 170, 2904, 1413, 1115, 8200, 1194, 124, 11739, 1105, 1103, 3487, 17917, 114, 117, 1110, 170, 3014, 117, 2030, 2576, 5921, 1104, 20

#### Prepare start and end positions of the answers and add to the inputs

For each chunk, we have to generate the true start position and end position. Because the dataset only gives `start char index of the answer in the context` of answer from the original sample along with the answer's text. Note that, start_positions, end_positions we generate here are token-level start and end positions of the answer , which are the labels the QA model will be trained to predict, which is at token level not char level.

Hence, we need to generate `start token index` and `end token index` to facilitate training. For this, we make use of `overflow_to_sample_mapping` and `offset_mapping`. Also be aware that sometimes, the whole answer may get separated in different chunks.(end token of chunk and start token of another chunk)

In [ ]:
ds["train"][0:3]["answers"] # true answer of first 3 samples

[{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]},
 {'text': ['a copper statue of Christ'], 'answer_start': [188]},
 {'text': ['the Main Building'], 'answer_start': [279]}]

In [ ]:
answers = ds["train"][0:3]["answers"]

In [ ]:
len(answers) # list(dicts(lists))

3

In [ ]:
# Test
a = [1, 2, 3, 4, 5, 5, 5]
idx = 0
while a[idx] != 5:
    print(a[idx])
    idx += 1

1
2
3
4


In [ ]:
# Prepare start and end positions
start_positions = []
end_positions = []

for chunk, offset in enumerate(inputs["offset_mapping"]):  # loop through the offset of each chunk
    original_sample_idx = inputs["overflow_to_sample_mapping"][chunk]
    print(f"{original_sample_idx=}, {chunk=}")
    print(f"{offset=}")

    # Get the true answer
    answers = ds["train"][original_sample_idx]["answers"]  # get the true answer and start char idx of each original sample
    answer_text = answers["text"][0]
    start_char = answers["answer_start"][0]  # answer`s start_char in the context for the whole original current sample
    # Calculate end_char
    end_char = start_char + len(answer_text)  # answer`s end_char in the context for the whole original current sample
    print(f"{answer_text=}, {start_char=}, {end_char=}")

    # Find out from which token idx the context starts and ends for each chunked input sequences (ques + context)
    sequence_ids = inputs.sequence_ids(chunk)  # [None, 0,...,0, 1,...,1, None]
    print(f"{sequence_ids=}")
    # Get the starting token index of the context in the current chunk
    idx = 0
    while sequence_ids[idx] != 1:
        idx += 1
    context_start = idx
    # Get the ending token index of the context in the current chunk
    while sequence_ids[idx] == 1:
        idx += 1
    context_end = idx - 1 # -1 cuz while loop increments the index before truth check
    print(f"{context_start=}, {context_end=}")
    # Prepare the labels in the format of (start_token_pos, end_token_pos) for each chunk
    # Case 1: If the context chunk does not fully contain the answer, label is (0, 0)
    print(f"{offset[context_start]=}, {offset[context_end]=}")
    if offset[context_start][0] > start_char or offset[context_end][1] < end_char:  # forward pass
        start_positions.append(0)
        end_positions.append(0)
    # Case 2: If the context chunk contains the start and end positions of that token
    else:
        # Find start position of the answer in the chunk
        idx = context_start  # start from beginning of context to forward until start char is reached
        while idx <= context_end and offset[idx][0] <= start_char:
            idx += 1
        start_positions.append(idx - 1)
        # Find end position of the answer in the chunk
        idx = context_end # start from end of context to backward until end char is reached
        while idx >= context_start and offset[idx][0] >= end_char:
            idx -= 1
        end_positions.append(idx + 1)
    print(f"{start_positions=}, {end_positions=}")
    print("..................................................")


original_sample_idx=0, chunk=0
offset=[(0, 0), (0, 2), (3, 7), (8, 11), (12, 15), (16, 22), (23, 27), (28, 37), (38, 44), (45, 47), (48, 52), (53, 55), (56, 59), (59, 63), (64, 70), (70, 71), (0, 0), (0, 13), (13, 15), (15, 16), (17, 20), (21, 27), (28, 31), (32, 33), (34, 42), (43, 52), (52, 53), (54, 56), (56, 58), (59, 62), (63, 67), (68, 76), (76, 77), (77, 78), (79, 83), (84, 88), (89, 91), (92, 93), (94, 100), (101, 107), (108, 110), (111, 114), (115, 121), (122, 126), (126, 127), (128, 139), (140, 142), (143, 148), (149, 151), (152, 155), (156, 160), (161, 169), (170, 173), (174, 180), (181, 183), (183, 184), (185, 187), (188, 189), (190, 196), (197, 203), (204, 206), (207, 213), (214, 218), (219, 223), (224, 226), (226, 229), (229, 232), (233, 237), (238, 241), (242, 248), (249, 250), (250, 251), (251, 254), (254, 256), (257, 259), (260, 262), (263, 264), (264, 265), (265, 268), (268, 269), (269, 270), (271, 275), (276, 278), (279, 282), (283, 287), (288, 296), (297, 299), (300

In [ ]:
start_positions, end_positions

([0, 0, 72, 40, 53, 17, 0, 0, 83, 51, 19, 0],
 [0, 0, 79, 47, 58, 22, 0, 0, 86, 54, 22, 0])

In [ ]:
ds["train"][0]["answers"]["text"][0]

'Saint Bernadette Soubirous'

In [ ]:
inputs["input_ids"][2] # 3rd chunk of sample 0

[101,
 1706,
 2292,
 1225,
 1103,
 6567,
 2090,
 9273,
 2845,
 1107,
 8109,
 1107,
 10111,
 20500,
 1699,
 136,
 102,
 5893,
 1106,
 1103,
 4304,
 4334,
 1110,
 1103,
 19349,
 1104,
 1103,
 11373,
 4641,
 119,
 13301,
 1481,
 1103,
 171,
 17506,
 9538,
 1110,
 1103,
 144,
 10595,
 2430,
 117,
 170,
 14789,
 1282,
 1104,
 8070,
 1105,
 9284,
 119,
 1135,
 1110,
 170,
 16498,
 1104,
 1103,
 176,
 10595,
 2430,
 1120,
 10111,
 20500,
 117,
 1699,
 1187,
 1103,
 6567,
 2090,
 25153,
 1193,
 1691,
 1106,
 2216,
 17666,
 6397,
 3786,
 1573,
 25422,
 13149,
 1107,
 8109,
 119,
 1335,
 1103,
 1322,
 1104,
 1103,
 1514,
 2797,
 113,
 1105,
 1107,
 170,
 2904,
 1413,
 1115,
 8200,
 1194,
 124,
 102]

In [ ]:
start_positions[2], end_positions[2] # start and end token position of 3rd chunk

(72, 79)

In [ ]:
sample_idx = inputs["overflow_to_sample_mapping"][2] # original smaple of 3rd chunk
sample_idx

0

In [ ]:
answers = ds["train"][0:3]["answers"] # first 3 samples
answers[sample_idx]["text"][0]

'Saint Bernadette Soubirous'

In [ ]:
inputs["input_ids"][2][72:79] # 3rd chunk of sample 0 with start to end token indices

[2216, 17666, 6397, 3786, 1573, 25422, 13149]

In [ ]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"][2][72:79])

['Saint', 'Bern', '##ade', '##tte', 'So', '##ubi', '##rous']

In [ ]:
tokenizer.decode(inputs["input_ids"][2][72:79])  # 3rd chunk of sample 0

'Saint Bernadette Soubirous'

In [ ]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"][3][40:47]) # 4th chunk of sample 0

['Saint', 'Bern', '##ade', '##tte', 'So', '##ubi', '##rous']

In [ ]:
tokenizer.decode(inputs["input_ids"][3][40:47]) # 4th chunk of sample 0

'Saint Bernadette Soubirous'

Here context_start and context_end are start and ends of each chunk.

- Case 1: Answer is NOT fully contained within the context span of the tokenized input.
    - `offset[context_start][0] > start_char`
        - This checks if the first character of the first token of the context in the tokenized sequence (offset[context_start][0]) comes after the start_char of the answer. If true, it means the answer starts before the context actually begins in this specific tokenized input chunk. This can happen when the original context is very long and gets truncated, and the answer falls outside the truncated window.
    - `offset[context_end][1] < end_char`
        - This checks if the last character of the last token of the context in the tokenized sequence (offset[context_end][1]) comes before the end_char of the answer. If true, it means the answer ends after the context ends in this tokenized input chunk (again, often due to truncation).
    - Result `(start_positions.append(0); end_positions.append(0))`
        - If either of these conditions is true, it means the answer span from SQuAD is not entirely present within the context tokens of this specific tokenized input. In such cases, a common convention is to set the start and end positions to 0 (or [CLS] token index), effectively telling the model "no answer found here" or "answer is not in this segment". The model's loss function (e.g., CrossEntropyLoss) for QA models is often designed to handle a 0,0 prediction for unanswerable questions or answers outside the span.

- Case 2: Answer is fully contained within the context span of the tokenized input.

    - This else block executes when the answer's full character span [start_char, end_char] is indeed covered by the character span of the tokenized context tokens `[offset[context_start][0], offset[context_end][1]]`.

    - Finding start_positions:

        - `idx = context_start`: Start searching from the first token of the context.
        - `while idx <= context_end and offset[idx][0] <= start_char`: This loop iterates forward through the context tokens. It continues as long as idx is within the bounds of the context tokens.The start character of the current token (offset[idx][0]) is less than or equal to the start_char of the answer. When the loop finishes, idx will point to the first token whose start character is GREATER than start_char of the answer.
        - `start_positions.append(idx - 1)`: Therefore, the actual start token should be the previous token (idx - 1), because that token's character span either starts at or before the answer's start_char and extends over it, or is the last token that completely precedes it. This effectively finds the earliest token that covers or immediately precedes the start_char of the answer.
    - Finding end_positions:

        - `idx = context_end`: Start searching backward from the last token of the context.
        - `while idx >= context_start and offset[idx][1] >= end_char`: This loop iterates backward through the context tokens. It continues as long as idx is within the bounds of the context tokens. The end character of the current token (offset[idx][1]) is greater than or equal to the end_char of the answer. When the loop finishes, idx will point to the first token whose end character is LESS than end_char of the answer.
        - `end_positions.append(idx + 1)`: Therefore, the actual end token should be the next token (idx + 1), because that token's character span either ends at or after the answer's end_char or is the first token that completely follows it. This effectively finds the latest token that covers or immediately follows the end_char of the answer.

**Why this approach is needed?**

Subword tokenization means that the exact character boundaries of your answer rarely align perfectly with the start and end of tokens. This logic ensures that the chosen start and end token indices form a span that minimally but fully encompasses the original answer's character span within the tokenized context. This is crucial for training extractive QA models, as they predict these token indices.

#### Final culminated preprocessing block

In [ ]:
ds["train"][0]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

In [ ]:
max_length = 384
stride = 128

def preprocess_training_examples(examples): # batched
    questions = [question.strip() for question in examples["question"]]
    tokenized_inputs = tokenizer(
        questions, examples["context"], max_length=max_length, truncation="only_second", stride=stride,
        return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length"
    )
    offset_mapping = tokenized_inputs.pop("offset_mapping")  # doesnt need by the model
    sample_map = tokenized_inputs.pop("overflow_to_sample_mapping") # doesnt need by the model
    answers = examples["answers"]
    # Get answers start and end token positions
    start_positions, end_positions = [], []
    for chunk, offset in enumerate(offset_mapping):
        # Get start char idx and end char idx of the answer in the context for each sample
        sample_idx = sample_map[chunk]
        answer = answers[sample_idx]
        answer_text = answer["text"][0]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer_text)
        # Get context start and end token idxs for each chunk
        sequence_ids = tokenized_inputs.sequence_ids(chunk)
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1
        # Prepare labels (start token idx of answer, end token idx of answer) for each chunk
        # Case 1: If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        # Case 2: Otherwise answer exists in this chunks and return the start and end token positions of the answer
        else:
            # start from beginning
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
            # start from end
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized_inputs["start_positions"] = start_positions
    tokenized_inputs["end_positions"] = end_positions
    return tokenized_inputs

# Apply preprocessing (only to train, we dont need start and end positions for validation sets as metric doenst need it)
train_ds = ds["train"].map(preprocess_training_examples, batched=True, remove_columns=ds["train"].column_names) # remove columns to avoid mismatched len errors cuz of overflow
train_ds

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 88729
})

In [ ]:
print(train_ds[0])

{'input_ids': [101, 1706, 2292, 1225, 1103, 6567, 2090, 9273, 2845, 1107, 8109, 1107, 10111, 20500, 1699, 136, 102, 22182, 1193, 117, 1103, 1278, 1144, 170, 2336, 1959, 119, 1335, 4184, 1103, 4304, 4334, 112, 188, 2284, 10945, 1110, 170, 5404, 5921, 1104, 1103, 6567, 2090, 119, 13301, 1107, 1524, 1104, 1103, 4304, 4334, 1105, 4749, 1122, 117, 1110, 170, 7335, 5921, 1104, 4028, 1114, 1739, 1146, 14089, 5591, 1114, 1103, 7051, 107, 159, 21462, 1566, 24930, 2508, 152, 1306, 3965, 107, 119, 5893, 1106, 1103, 4304, 4334, 1110, 1103, 19349, 1104, 1103, 11373, 4641, 119, 13301, 1481, 1103, 171, 17506, 9538, 1110, 1103, 144, 10595, 2430, 117, 170, 14789, 1282, 1104, 8070, 1105, 9284, 119, 1135, 1110, 170, 16498, 1104, 1103, 176, 10595, 2430, 1120, 10111, 20500, 117, 1699, 1187, 1103, 6567, 2090, 25153, 1193, 1691, 1106, 2216, 17666, 6397, 3786, 1573, 25422, 13149, 1107, 8109, 119, 1335, 1103, 1322, 1104, 1103, 1514, 2797, 113, 1105, 1107, 170, 2904, 1413, 1115, 8200, 1194, 124, 11739, 1105, 11

In [ ]:
# Apply preprocessing to validation dataset(we dont need start and end positions for validation sets)
# we need ids and offset_mapping for metric, but for model, we dont need these and have to be removed
def preprocess_validation_examples(examples):
    questions = [question.strip() for question in examples["question"]]
    tokenized_inputs = tokenizer(
        questions, examples["context"],
        max_length=max_length, stride=stride, truncation="only_second",
        return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length"
    )
    # Add ids and offsets to tokenized_inputs
    example_ids = []
    sample_map = tokenized_inputs.pop("overflow_to_sample_mapping")
    for i in range(len(tokenized_inputs["input_ids"])):
        offset_mappings = tokenized_inputs["offset_mapping"][i]
        sequence_ids = tokenized_inputs.sequence_ids(i)
        # Get only the offsets from context. We dont need offsets for questions here, so we make offset as None for question
        tokenized_inputs["offset_mapping"][i] = [offset if sequence_ids[idx] == 1 else None for idx, offset in enumerate(offset_mappings)]
        # Get the example id for tokenized inputs from the original smaples
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
    tokenized_inputs["example_id"] = example_ids
    return tokenized_inputs

validation_ds = ds["validation"].map(preprocess_validation_examples, batched=True, remove_columns=ds["validation"].column_names) # remove columns to avoid mismatched len errors cuz of overflow
validation_ds

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 10822
})

In [ ]:
print(validation_ds[0])

{'input_ids': [101, 5979, 4279, 1264, 2533, 1103, 10402, 1120, 3198, 5308, 1851, 136, 102, 3198, 5308, 1851, 1108, 1126, 1237, 1709, 1342, 1106, 4959, 1103, 3628, 1104, 1103, 1305, 2289, 1453, 113, 4279, 114, 1111, 1103, 1410, 1265, 119, 1109, 1237, 2289, 3047, 113, 10402, 114, 3628, 7068, 14722, 2378, 1103, 1305, 2289, 3047, 113, 24743, 114, 3628, 2938, 13598, 1572, 782, 1275, 1106, 7379, 1147, 1503, 3198, 5308, 1641, 119, 1109, 1342, 1108, 1307, 1113, 1428, 128, 117, 1446, 117, 1120, 12388, 112, 188, 3339, 1107, 1103, 1727, 2948, 2410, 3894, 1120, 3364, 10200, 117, 1756, 119, 1249, 1142, 1108, 1103, 13163, 3198, 5308, 117, 1103, 2074, 13463, 1103, 107, 5404, 5453, 107, 1114, 1672, 2284, 118, 12005, 11751, 117, 1112, 1218, 1112, 7818, 28117, 20080, 16264, 1103, 3904, 1104, 10505, 1296, 3198, 5308, 1342, 1114, 2264, 183, 15447, 16179, 113, 1223, 1134, 1103, 1342, 1156, 1138, 1151, 1227, 1112, 107, 3198, 5308, 149, 107, 114, 117, 1177, 1115, 1103, 7998, 1180, 15199, 2672, 1103, 4944, 18

### Post processing and Evaluation Metrics Exact Match and F1

For Question Answering (QA) models, especially **extractive QA** where the answer is a span of text from a given context (like SQuAD), the most commonly used and important metrics are:

1.  **Exact Match (EM)**
2.  **F1 Score**

These metrics are calculated by comparing the model's predicted answer string to the ground truth (reference) answer string(s).


### 1. Exact Match (EM)

* **Definition:** EM is a strict binary metric. It is `1` if the model's predicted answer string **exactly matches** one of the ground truth answer strings for a given question, and `0` otherwise.
* **Calculation:**
    * For each question:
        * Take the model's predicted answer string.
        * Compare it to *all* provided ground truth answers for that question (SQuAD often has multiple valid answers).
        * If the predicted answer is identical to *any* of the ground truth answers, the EM for that question is 1.
        * Otherwise, the EM is 0.
    * The overall EM score for the dataset is the **average EM score across all questions**.
* **Preprocessing for Comparison:** Before comparison, both the predicted and ground truth answer strings are typically "normalized" by:
    * Lowercasing.
    * Removing articles ("a", "an", "the").
    * Removing punctuation.
    * Standardizing whitespace.
    * This normalization helps to focus on the semantic correctness rather than trivial syntactic differences.
* **Pros:** Very intuitive and easy to understand.
* **Cons:** Very strict. It penalizes even minor differences. For example, if the true answer is "New York" and the model predicts "New York City", the EM score would be 0, even though it's semantically very close.

### 2. F1 Score

* **Definition:** F1 score is a more flexible metric than EM. It measures the **overlap** between the predicted answer and the ground truth answer at the word (or token) level. It is the harmonic mean of Precision and Recall.
* **Calculation (for a single question-answer pair):**
    1.  **Tokenize:** Both the predicted answer string and *each* of the ground truth answer strings are tokenized (usually by splitting on whitespace and normalizing punctuation/casing, similar to EM).
    2.  **Precision:**
        $\text{Precision} = \frac{\text{Number of shared tokens between predicted and true answer}}{\text{Total number of tokens in predicted answer}}$
    3.  **Recall:**
        $\text{Recall} = \frac{\text{Number of shared tokens between predicted and true answer}}{\text{Total number of tokens in ground truth answer}}$
    4.  **F1 Score:**
        $\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$
    5.  **Handling Multiple Ground Truths:** For a given question, there might be multiple valid ground truth answers. In this case, the F1 score for that question is the **maximum F1 score** calculated between the predicted answer and *any* of the ground truth answers.
    6.  **Overall F1 Score:** The overall F1 score for the dataset is the **average F1 score across all questions**.
* **Pros:** More forgiving than EM. It gives partial credit for partial matches. Better reflects semantic similarity when exact string matching is too harsh.
* **Cons:** Still relies on word overlap and doesn't fully capture paraphrasing or deeper semantic understanding. For example, "big" vs. "large" would be treated as different words unless handled by synonym lists, which are usually not part of the standard F1 calculation for SQuAD.

### How they are typically used:

* Both EM and F1 are reported side-by-side to give a comprehensive view of the model's performance.
* EM tells you how often the model is *perfectly* correct.
* F1 tells you how well the model *overlaps* with the correct answer, even if not perfectly exact.

### Example:

**Ground Truth Answers:** `["Stanford University", "Stanford"]`
**Predicted Answer:** `"Stanford University"`

* **Normalization (simple example: lowercase and remove punctuation):**
    * GT1: "stanford university"
    * GT2: "stanford"
    * Pred: "stanford university"

* **EM Calculation:**
    * "stanford university" == "stanford university" (GT1)? Yes.
    * **EM = 1** for this question.

* **F1 Calculation (comparing "stanford university" with GT1 "stanford university"):**
    * Tokens Pred: ["stanford", "university"]
    * Tokens GT1: ["stanford", "university"]
    * Shared: ["stanford", "university"] (2 tokens)
    * Precision = 2 / 2 = 1.0
    * Recall = 2 / 2 = 1.0
    * F1 = 2 * (1.0 * 1.0) / (1.0 + 1.0) = 1.0

* **F1 Calculation (comparing "stanford university" with GT2 "stanford"):**
    * Tokens Pred: ["stanford", "university"]
    * Tokens GT2: ["stanford"]
    * Shared: ["stanford"] (1 token)
    * Precision = 1 / 2 = 0.5
    * Recall = 1 / 1 = 1.0
    * F1 = 2 * (0.5 * 1.0) / (0.5 + 1.0) = 2 * 0.5 / 1.5 = 1 / 1.5 = 0.666...

* **Maximum F1:** Max(1.0, 0.666...) = 1.0. So, **F1 = 1.0** for this question.

**Another Example:**

**Ground Truth Answers:** `["the capital of France"]`
**Predicted Answer:** `"Paris, the capital"`

* **Normalization:**
    * GT: "the capital of france"
    * Pred: "paris the capital"

* **EM:** "paris the capital" == "the capital of france"? No.
    * **EM = 0**

* **F1 Calculation:**
    * Tokens Pred: ["paris", "the", "capital"]
    * Tokens GT: ["the", "capital", "of", "france"]
    * Shared: ["the", "capital"] (2 tokens)
    * Precision = 2 / 3 = 0.667
    * Recall = 2 / 4 = 0.5
    * F1 = 2 * (0.667 * 0.5) / (0.667 + 0.5) = 2 * 0.3335 / 1.167 = 0.667 / 1.167 = 0.572...
    * **F1 ≈ 0.57** for this question.

This illustrates how F1 gives partial credit even when EM is 0.

Hugging Face's `evaluate` library provides a convenient way to compute these metrics for SQuAD datasets, often wrapping the official SQuAD evaluation script.

More versatile metrics for Q and A Modes are

**Semantic Answer Similarity (SAS):**
SAS measures the semantic similarity between the predicted and correct answers, which is useful when dealing with paraphrases or answers that convey the same meaning but are worded differently. It can be calculated using techniques like cosine similarity on word embeddings or by training a separate model to predict answer similarity.

**Context Integrity and Answer Relevance:***
Context integrity refers to the model's ability to correctly identify when a question cannot be answered from the given context and avoid generating a response. Answer relevance assesses how well the predicted answer addresses the specific question, avoiding unnecessary details or irrelevant information.

**ROUGE:**
Measures the overlap between generated and reference texts, focusing on recall and text similarity.

```
Raw Text (Context + Question)
      |
      V
Tokenizer (produces input_ids, attention_mask, token_type_ids, offset_mapping)
      |
      V
QA Model (e.g., BERTForQuestionAnswering)
      |
      V
Start Logits & End Logits (scores for each token)
      |
      V
Post-processing Logic (Hugging Face's `run_qa.py` or `evaluate` library provides this):
    1. Iterate over all valid (start_idx, end_idx) pairs within context tokens.
    2. Score each span (e.g., start_logit + end_logit).
    3. Apply constraints (length, within context, etc.).
    4. Handle "no answer" predictions (SQuAD 2.0).
    5. Select the best span based on score.
    6. Use `offset_mapping` to extract the corresponding *string* from the original context.
      |
      V
Predicted Answer String
      |
      V
Evaluation Script (e.g., SQuAD evaluate-v1.1.py / v2.0.py or `evaluate` library):
    1. Normalize predicted and ground truth answer strings.
    2. Compute EM for each example.
    3. Compute F1 for each example (max over multiple ground truths).
    4. Average EM and F1 over the entire dataset.
      |
      V
Final EM and F1 Scores
```

In [ ]:
import evaluate

metric = evaluate.load("squad")

metric  # needs predictions: {"id": ..., "prediction_text": ...}, references: {"id": ..., "answers": ..., "answer_start": ...}

EvaluationModule(name: "squad", module_type: "metric", features: {'predictions': {'id': Value(dtype='string', id=None), 'prediction_text': Value(dtype='string', id=None)}, 'references': {'id': Value(dtype='string', id=None), 'answers': Sequence(feature={'text': Value(dtype='string', id=None), 'answer_start': Value(dtype='int32', id=None)}, length=-1, id=None)}}, usage: """
Computes SQuAD scores (F1 and EM).
Args:
    predictions: List of question-answers dictionaries with the following key-values:
        - 'id': id of the question-answer pair as given in the references (see below)
        - 'prediction_text': the text of the answer
    references: List of question-answers dictionaries with the following key-values:
        - 'id': id of the question-answer pair (see above),
        - 'answers': a Dict in the SQuAD dataset format
            {
                'text': list of possible texts for the answer, as a list of strings
                'answer_start': list of start positions for 

In [ ]:
# Lets see how we generate metrics on small validation set
small_eval_set = ds["validation"].select(range(100)) # sample val set

# Preprocess
eval_set = small_eval_set.map(preprocess_validation_examples, batched=True, remove_columns=ds["validation"].column_names)
eval_set = eval_set.remove_columns(["token_type_ids"])
eval_set_for_model = eval_set.remove_columns(["offset_mapping", "example_id"]) # Model dont expect these cols but we need them for metrics

small_eval_set, eval_set, eval_set_for_model

# Note the after preprocessing, the overflow tokens can sometimes results in more rows. Here luckily its 100


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

(Dataset({
     features: ['id', 'title', 'context', 'question', 'answers'],
     num_rows: 100
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'offset_mapping', 'example_id'],
     num_rows: 100
 }),
 Dataset({
     features: ['input_ids', 'attention_mask'],
     num_rows: 100
 }))

In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_checkpoint = "distilbert-base-cased-distilled-squad"
pretrained_model = AutoModelForQuestionAnswering.from_pretrained(trained_checkpoint)
pretrained_model.to(device)


eval_set_for_model.set_format("torch")
batch = {k: eval_set_for_model[k].to(device) for k in eval_set_for_model.column_names}

with torch.no_grad():
    outputs = pretrained_model(**batch)

outputs

QuestionAnsweringModelOutput(loss=None, start_logits=tensor([[ -2.2607,  -5.1783,  -5.2709,  ...,  -9.5243,  -9.5183,  -9.5288],
        [ -2.5961,  -5.5482,  -5.5313,  ...,  -9.9598,  -9.9533,  -9.9860],
        [ -3.7127,  -7.1848,  -8.5388,  ..., -11.6557, -11.6571, -11.6505],
        ...,
        [ -2.0260,  -4.4167,  -4.4980,  ...,  -8.1479,  -8.1530,  -8.1760],
        [ -4.1553,  -5.8304,  -7.1642,  ..., -10.5255, -10.5251, -10.4890],
        [ -3.2000,  -5.8162,  -6.7249,  ...,  -9.4935,  -9.5038,  -9.4871]],
       device='cuda:0'), end_logits=tensor([[ -0.7353,  -4.9236,  -5.1048,  ...,  -8.8734,  -8.8916,  -8.8550],
        [ -1.3056,  -5.3870,  -5.4945,  ...,  -9.4895,  -9.5039,  -9.4959],
        [ -2.7649,  -7.2201,  -9.0916,  ..., -11.3106, -11.3414, -11.2702],
        ...,
        [ -0.0768,  -4.8210,  -4.4374,  ...,  -8.0483,  -8.0502,  -7.9903],
        [ -2.7347,  -5.3650,  -7.2549,  ..., -10.0498, -10.0661,  -9.9886],
        [ -1.0991,  -4.2569,  -6.1267,  ...,  -8

In [ ]:
pretrained_model

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

In [ ]:
start_logits = outputs.start_logits.cpu().numpy()
end_logits = outputs.end_logits.cpu().numpy()

start_logits.shape, end_logits.shape  # (bs, seq_len) (bs, seq_len)

((100, 384), (100, 384))

In [ ]:
# Get the ids for each sample as metric expect this to know from sample the answer and true label belongs to
from collections import defaultdict

examples_to_ids = defaultdict(list)
for idx, sample in enumerate(eval_set):
    examples_to_ids[sample["example_id"]].append(idx)

examples_to_ids

defaultdict(list,
            {'56be4db0acb8001400a502ec': [0],
             '56be4db0acb8001400a502ed': [1],
             '56be4db0acb8001400a502ee': [2],
             '56be4db0acb8001400a502ef': [3],
             '56be4db0acb8001400a502f0': [4],
             '56be8e613aeaaa14008c90d1': [5],
             '56be8e613aeaaa14008c90d2': [6],
             '56be8e613aeaaa14008c90d3': [7],
             '56bea9923aeaaa14008c91b9': [8],
             '56bea9923aeaaa14008c91ba': [9],
             '56bea9923aeaaa14008c91bb': [10],
             '56beace93aeaaa14008c91df': [11],
             '56beace93aeaaa14008c91e0': [12],
             '56beace93aeaaa14008c91e1': [13],
             '56beace93aeaaa14008c91e2': [14],
             '56beace93aeaaa14008c91e3': [15],
             '56bf10f43aeaaa14008c94fd': [16],
             '56bf10f43aeaaa14008c94fe': [17],
             '56bf10f43aeaaa14008c94ff': [18],
             '56bf10f43aeaaa14008c9500': [19],
             '56bf10f43aeaaa14008c9501': [20],
     

In [ ]:
metric = evaluate.load("squad")
metric  # needs predictions: {"id": ..., "prediction_text": ...}, references: {"id": ..., "answers": ..., "answer_start": ...}

EvaluationModule(name: "squad", module_type: "metric", features: {'predictions': {'id': Value(dtype='string', id=None), 'prediction_text': Value(dtype='string', id=None)}, 'references': {'id': Value(dtype='string', id=None), 'answers': Sequence(feature={'text': Value(dtype='string', id=None), 'answer_start': Value(dtype='int32', id=None)}, length=-1, id=None)}}, usage: """
Computes SQuAD scores (F1 and EM).
Args:
    predictions: List of question-answers dictionaries with the following key-values:
        - 'id': id of the question-answer pair as given in the references (see below)
        - 'prediction_text': the text of the answer
    references: List of question-answers dictionaries with the following key-values:
        - 'id': id of the question-answer pair (see above),
        - 'answers': a Dict in the SQuAD dataset format
            {
                'text': list of possible texts for the answer, as a list of strings
                'answer_start': list of start positions for 

In [ ]:
small_eval_set, eval_set

(Dataset({
     features: ['id', 'title', 'context', 'question', 'answers'],
     num_rows: 100
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'offset_mapping', 'example_id'],
     num_rows: 100
 }))

In [ ]:
# Generate predictions with prediction text and id i.e token idx of start and end tokens
import numpy as np
predictions = []
n_best = 20 # get first best 30 answers for each example
max_answer_length = 30

for i, sample in enumerate(small_eval_set):
    example_id = sample["id"]
    context = sample["context"]
    answers = []

    for idx in examples_to_ids[example_id]:
        start_logits = outputs.start_logits[idx].cpu().numpy()  # max_seq_len
        end_logits = outputs.end_logits[idx].cpu().numpy()      # max_seq_len
        offsets = eval_set["offset_mapping"][idx]

        # get the prediction text
        start_indexes = np.argsort(start_logits)[-1 : -n_best - 1 : -1].tolist()  # get n-best logits [start : stop : step]
        end_indexes = np.argsort(end_logits)[-1 : -n_best - 1 : -1].tolist()  # get n-best logits
        for start_idx in start_indexes:
            for end_idx in end_indexes:
                # skip answers that are not fully in the context
                if offsets[start_idx] is None or offsets[end_idx] is None:
                    continue
                # skip answers with invalid length
                if (end_idx < start_idx or end_idx - start_idx + 1 > max_answer_length):
                    continue
                answers.append(
                    {
                        "text": context[offsets[start_idx][0] : offsets[end_idx][1]],
                        "logit_score": start_logits[start_idx] + end_logits[end_idx]
                    }
                )
    best_answer = max(answers, key=lambda x: x["logit_score"])
    predictions.append({"id": example_id, "prediction_text": best_answer["text"]})

This code takes the raw scores (logits) from a QA model and turns them into actual text answers.

Here's a simple breakdown:

1.  **Loop through each question-context example:** It processes one question at a time.

2.  **Get Logits & Offsets for each chunk:** A long context might be split into several "chunks." For each chunk, it gets the model's `start_logit` (score for each token being the start) and `end_logit` (score for each token being the end), plus `offsets` (character positions of each token in the original text).

3.  **Find Top Start/End Candidates:**
    * `np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()`: This finds the `n_best` tokens with the highest `start_logit` scores.
    * Similarly for `end_logit`.
    * For eg: [start: stop: step] When n_best is, say, 20: [-1 : -21 : -1]
        * -1 is the last element (the highest logit score's index).
        -n_best - 1 becomes -20 - 1 = -21.
        -1 is the step (meaning go backward by one element).
        This slice [-1 : -21 : -1] works perfectly fine. It tells NumPy to start at the last element, go backward one by one, until it reaches (but doesn't include) the element at index -21. This correctly gives you the top n_best elements in descending order of their scores (or rather, their original indices, which correspond to the highest scores).

4.  **Combine Candidates & Score Spans:**
    * It then tries every combination of these top `n_best` start and end tokens.
    * For each combination, it calculates a `logit_score` by adding the start and end logits.

5.  **Filter Invalid Spans:**
    * **Not in context:** Skips if the token doesn't map to a real part of the original context (e.g., `[CLS]`, `[SEP]`).
    * **Invalid Order:** Skips if the end token comes before the start token.
    * **Too Long:** Skips if the answer span is longer than `max_answer_length`.

6.  **Extract Text and Store:**
    * For valid spans, it uses the `offsets` to pull the actual answer `text` (characters) from the `context` string.
    * It stores this `text` and its `logit_score` in an `answers` list.
    

7.  **Select the Best Answer:** After checking all valid combinations for *all chunks* related to the current example, it picks the `best_answer` from the `answers` list based on the highest `logit_score`.

In short, It finds the most probable start and end token positions within the context chunks, extracts the corresponding text, filters out bad answers, and picks the highest-scoring one.

In [ ]:
predictions

[{'id': '56be4db0acb8001400a502ec', 'prediction_text': 'Denver Broncos'},
 {'id': '56be4db0acb8001400a502ed', 'prediction_text': 'Carolina Panthers'},
 {'id': '56be4db0acb8001400a502ee',
  'prediction_text': "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California"},
 {'id': '56be4db0acb8001400a502ef', 'prediction_text': 'Carolina Panthers'},
 {'id': '56be4db0acb8001400a502f0', 'prediction_text': 'gold'},
 {'id': '56be8e613aeaaa14008c90d1', 'prediction_text': 'golden anniversary'},
 {'id': '56be8e613aeaaa14008c90d2', 'prediction_text': 'February 7, 2016'},
 {'id': '56be8e613aeaaa14008c90d3',
  'prediction_text': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference'},
 {'id': '56bea9923aeaaa14008c91b9', 'prediction_text': 'golden anniversary'},
 {'id': '56bea9923aeaaa14008c91ba',
  'prediction_text': 'American Football Conference'},
 {'id': '56bea9923aeaaa14008c9

In [ ]:
len(predictions)

100

In [ ]:
small_eval_set["answers"][0]

{'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],
 'answer_start': [177, 177, 177]}

In [ ]:
# Get true labels
true_labels = [{"answers": example["answers"] , "id": example["id"]} for example in small_eval_set] # we need list[dicts]

In [ ]:
print(predictions[0])
print(true_labels[0])

{'id': '56be4db0acb8001400a502ec', 'prediction_text': 'Denver Broncos'}
{'answers': {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'], 'answer_start': [177, 177, 177]}, 'id': '56be4db0acb8001400a502ec'}


In [ ]:
metric.compute(predictions=predictions, references=true_labels)

{'exact_match': 83.0, 'f1': 88.25000000000004}

That’s rather good considering that according to its paper DistilBERT fine-tuned on SQuAD obtains 79.1 and 86.9 for those scores on the whole dataset.

Now let’s put everything we just did in a `compute_metrics()` function that we can use in the HF Trainer. Normally, that `compute_metrics()` function only receives a tuple `eval_preds` with `logits` and `labels` when using `Trainer`. Here we will need a bit more, as we have to look in the preprocessed dataset for the offset and the original dataset  for the original contexts, so we won’t be able to use this function to get regular evaluation results during training. We will only use it at the end of training to check the results.

The `compute_metrics()` function groups the same steps as before; we just add a small check in case we don’t come up with any valid answers (in which case we predict an empty string).

In [ ]:
# Final metric calculation
import evaluate

from collections import defaultdict

import numpy as np

from tqdm.auto import tqdm

metric = evaluate.load("squad")

def compute_metrics(start_logits, end_logits, original_val_ds, preprocessed_val_ds):
    n_best = 20
    max_answer_length = 30
    predictions = []
    true_labels = []

    example_to_ids = defaultdict(list)
    for feature_idx, sample in enumerate(preprocessed_val_ds):  # input_ids', 'attention_mask', 'offset_mapping', 'example_id'
        example_to_ids[sample["example_id"]].append(feature_idx)  # each sample can have more idxs for same example_id cuz of overflow. See feature idx as row idx

    for sample in tqdm(original_val_ds): # 'id', 'title', 'context', 'question', 'answers'
        id = sample["id"]
        context = sample["context"]
        predicted_answers = []

        # Loop through all features/rows/overflows associated with that sample
        for feature_idx in example_to_ids[id]:  # for each sample, we can have more than one idxs for same example_id cuz of overflow, this is row idx for sample in preprocessedvalds
            start_logits_feature = start_logits[feature_idx]
            end_logits_feature = end_logits[feature_idx]
            nbest_start_logits_idxs = np.argsort(start_logits_feature)[-1 : -n_best - 1: -1].tolist()
            nbest_end_logits_idxs = np.argsort(end_logits_feature)[-1 : -n_best - 1: -1].tolist()

            offsets = preprocessed_val_ds[feature_idx]["offset_mapping"]
            # Loop through n-best logits
            for start_logit_idx in nbest_start_logits_idxs:
                for end_logit_idx in nbest_end_logits_idxs:
                    # Skip answers with a length that is either < 0 or > max_answer_length.
                    if (end_logit_idx < start_logit_idx or end_logit_idx - start_logit_idx + 1 > max_answer_length):
                        continue
                    # Skip answers that are not fully in the context
                    if offsets[start_logit_idx] is None or offsets[end_logit_idx] is None:
                        continue
                    predicted_answers.append({
                        "text": context[offsets[start_logit_idx][0] : offsets[end_logit_idx][1]],
                        "logit_score": start_logits_feature[start_logit_idx] + end_logits_feature[end_logit_idx]  # aggregate both scores
                    })
        # Select the answer with the best logit score
        if len(predicted_answers) > 0:
            best_answer = max(predicted_answers, key=lambda x: x["logit_score"])
            predictions.append({"id": id, "prediction_text":best_answer["text"]})
        else:
            predictions.append({"id": id, "prediction_text": ""})

        true_labels.append({"id": id, "answers": sample["answers"]})

    results = metric.compute(predictions=predictions, references=true_labels)
    return results

In [ ]:
start_logits.shape, end_logits.shape

((384,), (384,))

In [ ]:
# Test
compute_metrics(start_logits, end_logits, small_eval_set, eval_set)

  0%|          | 0/100 [00:00<?, ?it/s]

{'exact_match': 86.0, 'f1': 90.75321637426902}

### DataLoaders

In [ ]:
import torch
from torch.utils.data import DataLoader

from transformers import default_data_collator

train_ds.set_format("torch")
validation_set = validation_ds.remove_columns(["example_id", "offset_mapping"]) # not needed for our model training
validation_set.set_format("torch")

train_dataloader = DataLoader(train_ds, shuffle=True, batch_size=8, collate_fn=default_data_collator)
eval_dataloader = DataLoader(validation_set, batch_size=8, collate_fn=default_data_collator)

In [ ]:
for batch in train_dataloader:
    break
batch

{'input_ids': tensor([[  101,  3986, 10138,  ...,     0,     0,     0],
         [  101,  1332,  1225,  ...,     0,     0,     0],
         [  101,  1731,  1242,  ...,     0,     0,     0],
         ...,
         [  101,  2777,  1225,  ...,     0,     0,     0],
         [  101,  1327,  1214,  ...,     0,     0,     0],
         [  101,  1130,  1155,  ...,     0,     0,     0]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]]),
 'start_positions': tensor([ 43, 118,  68,  10,  25, 210,  26,  65]),
 'end_positions': tensor([ 43, 119,  71,  12,  25, 213,  26

In [ ]:
for batch in eval_dataloader:
    break
batch

{'input_ids': tensor([[ 101, 5979, 4279,  ...,    0,    0,    0],
         [ 101, 5979, 4279,  ...,    0,    0,    0],
         [ 101, 2777, 1225,  ...,    0,    0,    0],
         ...,
         [ 101, 1327, 1108,  ...,    0,    0,    0],
         [ 101, 1327, 1285,  ...,    0,    0,    0],
         [ 101, 1327, 1110,  ...,    0,    0,    0]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]])}

### Model

In [ ]:
from transformers import AutoModelForQuestionAnswering

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mdl_checkpoint = "huggingface-course/bert-finetuned-squad"
model = AutoModelForQuestionAnswering.from_pretrained(mdl_checkpoint)
model.to(device)

model

BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, 

### Optimizer

In [ ]:
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=2e-5)

### Accelator

Once we have all those objects, we can send them to the accelerator.prepare() method. Remember that if you want to train on TPUs in a Colab notebook, you will need to move all of this code into a training function, and that shouldn’t execute any cell that instantiates an Accelerator(like len(traindataloader)). We can force mixed-precision training by passing fp16=True to the Accelerator as it can speed up the training nicely on a recent GPU.

As you should know, we can only use the train_dataloader length to compute the number of training steps after it has gone through the accelerator.prepare() method. We will the linear scheduler.

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator(mixed_precision="fp16")

model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(model, optimizer, train_dataloader, eval_dataloader)

In [ ]:
# Test
for batch in train_dataloader:
    break

outputs = model(**batch)

outputs, outputs.start_logits.shape, outputs.end_logits.shape

(QuestionAnsweringModelOutput(loss=tensor(1.0927, device='cuda:0', grad_fn=<DivBackward0>), start_logits=tensor([[ -7.7539, -10.3672, -10.4219,  ..., -10.9219, -10.9219, -10.9062],
         [-10.4453, -10.3984, -10.8203,  ..., -11.0391, -10.9922, -10.9531],
         [ -9.9062, -10.6953, -10.9375,  ..., -11.0469, -11.0625, -10.9531],
         ...,
         [ -9.4766,  -8.7656, -10.7734,  ..., -11.0547, -11.0703, -11.0469],
         [ -1.8066, -10.7578, -10.7266,  ..., -10.1562, -11.0078, -10.0312],
         [ -9.7812, -10.5156, -11.2578,  ..., -11.2031, -11.1641, -11.1641]],
        device='cuda:0', grad_fn=<ToCopyBackward0>), end_logits=tensor([[ -7.5391, -10.7188, -10.6094,  ..., -10.8984, -10.8984, -10.9453],
         [-10.3672, -10.6406, -10.6328,  ..., -10.8203, -10.8750, -10.9062],
         [ -9.4062, -10.7422, -10.7891,  ..., -10.8672, -10.8750, -10.9766],
         ...,
         [ -9.4297,  -5.3555, -10.9688,  ..., -10.8828, -10.8672, -10.8906],
         [ -1.8740, -10.5469, -10.

### LR scheduler

In [ ]:
from transformers import get_scheduler

num_batches = len(train_dataloader)  # num of steps per epoch
num_epochs = 3
num_training_steps = num_batches * num_epochs

lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_training_steps=num_training_steps, num_warmup_steps=0)

### Saving Checkpoints to Hf

In [ ]:
from huggingface_hub import HfApi, create_repo, upload_folder, get_full_repo_name
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")
api = HfApi(token=hf_token)

model_name = "bert-finetuned-squad-accelerate"
repo_id = get_full_repo_name(model_name, token=api.token)
print(repo_id)

output_dir = "bert-finetuned-squad-accelerate"
os.makedirs(output_dir, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "true"
try:
    create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True, token=api.token)
except Exception as e:
    print(f"Could not create/check repo {repo_id}: {e}")

BhavanaMalla/bert-finetuned-squad-accelerate


We can now upload anything we save in `output_dir` by calling the `upload_folder()` method. This will help us upload the intermediate models at the end of each epoch.

### Training

We are now ready to write the full training loop. After defining a progress bar to follow how training goes, the loop has four parts:

- The training in itself, which is the classic iteration over the train_dataloader
- Forward pass through the model
- Backward pass
- Optimizer step.

In the evaluation, we gather all the values for start_logits and end_logits before converting them to NumPy arrays. Once the evaluation loop is finished, we concatenate all the results.

Note: We need to truncate because the Accelerator may have added a few samples at the end to ensure we have the same number of examples in each process.

Saving and uploading, where we first save the model and the tokenizer, then call `upload_folder`

(repo.push_to_hub() is deprecated. Here use the argument blocking=False to tell the 🤗 Hub library to push in an asynchronous process. This way, training continues normally and this (long) instruction is executed in the background.)

In [ ]:
import tqdm
from tqdm.auto import tqdm
import numpy as np
import os

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_epochs):
    model.train()
    accelerator.print("Training")
    for step, batch in enumerate(train_dataloader): # 'input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)
    # Evaluation
    model.eval()
    accelerator.print("Validation")
    start_logits, end_logits = [], []
    for batch in tqdm(eval_dataloader): # 'input_ids', 'attention_mask'
        with torch.no_grad():
            outputs = model(**batch)
        start_logits.append(accelerator.gather(outputs.start_logits).cpu().numpy())  # (bs, max_seq_len)
        end_logits.append(accelerator.gather(outputs.end_logits).cpu().numpy())  # (bs, max_seq_len)

    start_logits = np.concatenate(start_logits)
    end_logits = np.concatenate(end_logits)
    # Truncation cuz of accelerator adding up random examples at end
    start_logits = start_logits[: len(validation_ds)]
    end_logits = end_logits[: len(validation_ds)]
    # Metrics
    metrics = compute_metrics(start_logits, end_logits, ds["validation"], validation_ds)  # ('id', 'title', 'context', 'question', 'answers') ('input_ids', 'attention_mask', 'offset_mapping', 'example_id')
    print(f"Epoch: {epoch}", metrics)
    # Save and upload to HF
    accelerator.wait_for_everyone()  #  to make sure we have the same model in every process before saving
    model_unwrapped = accelerator.unwrap_model(model)
    model_unwrapped.save_pretrained(output_dir, save_function=accelerator.save)
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)
        try:
            upload_folder(repo_id=repo_id, folder_path=output_dir, repo_type="model", commit_message=f"Training in progress epoch {epoch}", token=api.token, )
        except Exception as e:
            print(f"Failed to upload files: {e}")



  0%|          | 0/33276 [00:00<?, ?it/s]

Training
Validation


  0%|          | 0/1353 [00:00<?, ?it/s]

  0%|          | 0/10570 [00:00<?, ?it/s]

Epoch: 0 {'exact_match': 80.45411542100284, 'f1': 88.10997448417402}


model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

... (6 outputs left out)

  0%|          | 0/1353 [00:00<?, ?it/s]

  0%|          | 0/10570 [00:00<?, ?it/s]

Epoch: 2 {'exact_match': 79.99053926206244, 'f1': 87.96662464103616}


model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

Note: We grab the unwrapped_model, which is the base model we defined. The accelerator.prepare() method changes the model to work in distributed training, so it won’t have the save_pretrained() method anymore; the accelerator.unwrap_model() method undoes that step. Lastly, we call save_pretrained() but tell that method to use accelerator.save() instead of torch.save().

### Inference

In [ ]:
from transformers import pipeline

model_checkpoint = "BhavanaMalla/bert-finetuned-squad-accelerate"
question_answerer = pipeline("question-answering", model=model_checkpoint)

context = """
🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""
question = "Which deep learning libraries back 🤗 Transformers?"

question_answerer(question=question, context=context)

config.json:   0%|          | 0.00/633 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


{'score': 0.9982061982154846,
 'start': 78,
 'end': 105,
 'answer': 'Jax, PyTorch and TensorFlow'}

In [ ]:
import datasets
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, default_data_collator, get_scheduler, pipeline

import evaluate
from collections import defaultdict

import torch
from torch.utils.data import DataLoader
import torch.optim as optim

from accelerate import Accelerator

from huggingface_hub import HfApi, get_full_repo_name, create_repo, upload_folder
from google.colab import userdata

from tqdm.auto import tqdm

import numpy as np
import os

# Load Dataset
ds = load_dataset("squad")

# Preprocessing
checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
max_length = 384
stride = 128

def preprocess_training_data(examples):
    questions = [question.strip() for question in examples["question"]]
    context = examples["context"]
    tokenized_examples = tokenizer(questions, context, max_length=max_length, truncation="only_second", stride=stride,
                                   return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    overflow_sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offsets = tokenized_examples.pop("offset_mapping")
    answers = examples["answers"]
    # Get start_positions and end_positions
    start_positions = []
    end_positions = []
    for chunk_idx, offset in enumerate(offsets):
        original_sample_idx = overflow_sample_mapping[chunk_idx]
        answer = answers[original_sample_idx]
        answer_text = answer["text"][0]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer_text)
        sequence = tokenized_examples.sequence_ids(chunk_idx)
        # Get the context start and end at token level
        idx = 0
        while sequence[idx] != 1:
            idx += 1
        context_start = idx # token level
        while sequence[idx] == 1:
            idx += 1
        context_end = idx - 1  # token level
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char: # offset is char idxs at token level
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start # token level
            # Start from beginning and loop until we hit the start char idx in the offset of token
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)  # token idxs
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)
    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions
    return tokenized_examples

def preprocess_validation_data(examples):
    questions = [question.strip() for question in examples["question"]]
    context = examples["context"]
    tokenized_examples = tokenizer(questions, context, max_length=max_length, truncation="only_second", stride=stride,
                                   return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    overflow_sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")

    example_ids = []
    # Get offsets only for context and id for all samples
    for chunk_idx in range(len(tokenized_examples["input_ids"])):
        offset_mapping = tokenized_examples["offset_mapping"][chunk_idx]
        sequence_ids = tokenized_examples.sequence_ids(chunk_idx)
        # Get id for all chunks of respective sample
        sample_idx = overflow_sample_mapping[chunk_idx]
        example_id = examples["id"][sample_idx]
        example_ids.append(example_id)
        # Get offsets only for context
        offsets = [offset if sequence_ids[i] == 1 else None for i, offset in enumerate(offset_mapping)]
        tokenized_examples["offset_mapping"][chunk_idx] = offsets
    tokenized_examples["example_id"] = example_ids
    return tokenized_examples

train_sub = ds["train"] #.select(range(100))
val_sub = ds["validation"] # .select(range(20))
train_ds = train_sub.map(preprocess_training_data, batched=True, remove_columns=ds["train"].column_names)
validation_ds = val_sub.map(preprocess_validation_data, batched=True, remove_columns=ds["validation"].column_names)
validation_set = validation_ds.remove_columns(["offset_mapping", "example_id"])
validation_set.set_format("torch")

# DataLoaders
train_dataloader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=default_data_collator)
validation_dataloader = DataLoader(validation_set, batch_size=8, collate_fn=default_data_collator)

metric = evaluate.load("squad")

def compute_metric(start_logits, end_logits, original_val_ds, preprocessed_val):
    n_best = 20
    max_answer_length = 30

    example_to_ids = defaultdict(list)
    for idx, sample in enumerate(preprocessed_val):
        example_to_ids[sample["example_id"]].append(idx)

    predictions = []
    true_labels = []
    for sample in tqdm(original_val_ds):
        example_id = sample["id"]
        context = sample["context"]
        predicted_answers = []
        for idx in example_to_ids[example_id]:
            start_logit_idx = start_logits[idx]  # (384,)
            end_logit_idx = end_logits[idx]  # (384,)
            n_best_start_logits_idxs = np.argsort(start_logit_idx)[-1: -n_best: -1]  # (n_best,)
            n_best_end_logits_idxs = np.argsort(end_logit_idx)[-1: -n_best: -1]  # # (n_best,)
            offsets = preprocessed_val[idx]["offset_mapping"]
            for start_logit in n_best_start_logits_idxs:
                for end_logit in n_best_end_logits_idxs:
                    if offsets[start_logit] is None or offsets[end_logit] is None:
                        continue
                    if (end_logit > start_logit or end_logit - start_logit + 1 > max_answer_length):
                        continue
                    text = context[offsets[start_logit][0] : offsets[end_logit][1]]
                    predicted_answers.append({"text": text,
                                              "logit_score": start_logit + end_logit})
        if len(predicted_answers) > 0:
            best = max(predicted_answers, key=lambda x: x["logit_score"])
            predictions.append({"id": example_id, "prediction_text": best["text"]})
        else:
            predictions.append({"id": example_id, "prediction_text": ""})
        true_labels.append({"id": example_id, "answers": sample["answers"]})
    results = metric.compute(predictions=predictions, references=true_labels)
    return results

# Model
model_checkpoint = "huggingface-course/bert-finetuned-squad"
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
device = ("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

# Accelerator
accelerator = Accelerator(mixed_precision="fp16")
model, optimizer, train_dataloader, validation_dataloader = accelerator.prepare(model, optimizer, train_dataloader, validation_dataloader)

# LR scheduler
num_steps_per_epoch = len(train_dataloader)
num_epochs = 1
num_training_steps = num_epochs * num_steps_per_epoch
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Saving
api = HfApi()
hf_token = userdata.get("HF_TOKEN")
model_name = "bert-finetuned-squad-accelerate-self"
repo_id = get_full_repo_name(model_name)

output_dir = "bert-finetuned-squad-accelerate-self"
os.makedirs(output_dir, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

try:
    create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True, token=api.token)
except Exception as e:
    print("Repo createion failed: ", e)

# Training
progress_bar = tqdm(range(num_training_steps))
for epoch in range(num_epochs):
    accelerator.print("Training")
    model.train()
    for batch in train_dataloader:
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)

    accelerator.print("Validation")
    model.eval()
    start_logits, end_logits = [], []
    for batch in validation_dataloader:
        with torch.no_grad():
            outputs = model(**batch)

        start_logits.append(accelerator.gather(outputs.start_logits).cpu().numpy())
        end_logits.append(accelerator.gather(outputs.end_logits).cpu().numpy())
    start_logits = np.concatenate(start_logits)
    end_logits = np.concatenate(end_logits)
    results = compute_metric(start_logits, end_logits, ds["validation"], validation_ds)
    print(f"Epoch: {epoch}", results)

accelerator.wait_for_everyone()
unwrapped_model = accelerator.unwrap_model(model)
unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)

if accelerator.is_main_process:
    tokenizer.save_pretrained(output_dir)
    try:
        upload_folder(repo_id=repo_id, folder_path=output_dir, repo_type="model", commit_message=f"Training in progress epoch {epoch}", token=api.token)
    except Exception as e:
        print(f"Failed to upload files: {e}")


# Inference
trained_model = pipeline("question-answering", model=repo_id)

context = """
🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""

question = "Which deep learning libraries back 🤗 Transformers?"
trained_model(question=question, context=context)

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

... (8 outputs left out)

  0%|          | 0/11092 [00:00<?, ?it/s]

Training
Validation


  0%|          | 0/10570 [00:00<?, ?it/s]

Exception ignored in: <function _xla_gc_callback at 0x7e4965b01d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
Exception ignored in: <function _xla_gc_callback at 0x7e4965b01d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
Exception ignored in: <function _xla_gc_callback at 0x7e4965b01d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
Exception ignored in: <function _xla_gc_callback at 0x7e4965b01d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callbac